## Setup

Keep [`example_data.py`](example_data.py) and [`privacy_metrics.py`](privacy_metrics.py) beside this notebook.

In [1]:
#| label: setup
from IPython.display import display

from example_data import METRIC_CATEGORICAL_COLUMNS, make_example_views
from privacy_metrics import distance_membership_auc, distance_membership_details

## 1. Metric and inputs

| Property | Value |
|---|---|
| Metric | DCR membership-inference AUC |
| Family | Privacy |
| Origin | Established and adapted in the source project |
| Classifier | None; distance-based attack |
| Released table | $\widetilde Y$ |
| Members | rows of $\widetilde X$ |
| Non-members | independent rows of $\widetilde H^{\mathrm{miss}}$ |
| Range | $[0,1]$; $0.5$ means no membership ranking signal |

The attack asks whether training members are systematically closer to the released table than unseen non-members are.

## 2. Mixed-type record distance

Let $N$ be the numerical columns and $C$ the categorical columns of a complete representation with $q=|N|+|C|$ columns. Numerical values are already on the shared scale. For records $r$ and $y$, define

$$
d(r,y)=\frac1q\left[
\sum_{j\in N}|r_j-y_j|+
\sum_{j\in C}\mathbf1\{r_j\ne y_j\}
\right].
$$

The distance of record $r$ to the released table is

$$
d_Y(r)=\min_{y\in\widetilde Y}d(r,y).
$$

A smaller value means that $r$ has a closer representative in $Y$.

## 3. Membership score and AUC

Assign membership label $L=1$ to rows from $\widetilde X$ and $L=0$ to rows from $\widetilde H^{\mathrm{miss}}$. The attack score is negative distance:

$$
s(r)=-d_Y(r).
$$

The metric is the ROC AUC of $s$. Equivalently, for a random member $x$ and non-member $h$,

$$
D_{\mathrm{DCR-MIA}}(X,Y,H^{\mathrm{miss}})=
P(d_Y(x)<d_Y(h))+\frac12P(d_Y(x)=d_Y(h)).
$$

AUC $0.5$ means that distance cannot rank members above non-members. AUC $1$ means every sampled member is closer to $Y$ than every sampled non-member.

In [2]:
#| label: dcr-membership

example = make_example_views()
X_tilde = example.X_tilde
Y_tilde = example.Y_tilde
H_miss_tilde = example.H_miss_tilde

details = distance_membership_details(
    X_tilde,
    H_miss_tilde,
    Y_tilde,
    categorical_columns=METRIC_CATEGORICAL_COLUMNS,
)
D_dcr = distance_membership_auc(
    X_tilde,
    H_miss_tilde,
    Y_tilde,
    categorical_columns=METRIC_CATEGORICAL_COLUMNS,
)

display(details.round(4))
display(details.groupby("group")["closest_distance"].agg(["count", "mean", "median"]).round(4))
print(f"DCR membership-inference AUC = {D_dcr:.6f}")

,group,label,closest_distance,attack_score
0,member,1.0,0.0131,-0.0131
1,member,1.0,0.2312,-0.2312
2,member,1.0,0.1780,-0.1780
3,member,1.0,0.0093,-0.0093
4,member,1.0,0.0177,-0.0177
5,member,1.0,0.1844,-0.1844
6,non-member,0.0,0.0075,-0.0075
7,non-member,0.0,0.3773,-0.3773
8,non-member,0.0,0.1862,-0.1862
9,non-member,0.0,0.2006,-0.2006


,count,mean,median
group,,,
member,6,0.1056,0.0978
non-member,10,0.1561,0.1816


DCR membership-inference AUC = 0.633333


## 4. Interpretation

The shared example gives an AUC above $0.5$, so members tend to lie somewhat closer to $Y$ than independent non-members do. With such small tables, treat the number only as a transparent calculation.

Unlike exact-match rate, this metric detects near copies. Its roles are directional: $X$ supplies members, $Y$ is the released table, and $H^{\mathrm{miss}}$ supplies non-members. Swapping them changes the question. Values below $0.5$ indicate reversed ranking rather than stronger membership leakage.

## Takeaway

Measure each member's and non-member's distance to its closest released row, negate the distances to form attack scores, and report their ROC AUC.